# 02 - Cotizaciones Basicas

Este notebook prueba las funcionalidades basicas de cotizaciones de la API de IOL.

## Funcionalidades:
- Dolar MEP con diferentes bonos
- Cotizaciones de acciones (modelos tipados y JSON)
- Cotizaciones con diferentes mercados y plazos
- Datos basicos de titulos
- Opciones de titulos (Calls y Puts)
- Instrumentos por pais

**Nota:** Requiere credenciales validas de IOL en el archivo `.env`

## Configuracion Inicial

In [ ]:
import sys
import os
from datetime import datetime
import json
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath('../..'))

from pyIol import (
    IOLClient, IOLAPIError,
    CotizacionTitulo, Punta, DatosTitulo, OpcionTitulo, InstrumentoPais,
    Markets, SettlementTerms
)
print("Librerias importadas correctamente")
print(f"Fecha y hora: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Mercados disponibles: {[attr for attr in dir(Markets) if not attr.startswith('_')]}")

In [ ]:
load_dotenv('../../.env')
USERNAME = os.getenv('IOL_USERNAME', 'tu_usuario_iol')
PASSWORD = os.getenv('IOL_PASSWORD', 'tu_password_iol')

if USERNAME == "tu_usuario_iol" or PASSWORD == "tu_password_iol":
    print("ADVERTENCIA: Configura las credenciales en .env")
else:
    print(f"Credenciales configuradas - Usuario: {USERNAME}")

In [ ]:
try:
    client = IOLClient(USERNAME, PASSWORD)
    print("Cliente IOL creado correctamente")
except Exception as e:
    print(f"Error al crear cliente: {e}")
    client = None

## 1. Dolar MEP

In [ ]:
# Dolar MEP con AL30
if client:
    try:
        print("Obteniendo cotizacion del dolar MEP (AL30)...")
        mep_data = client.get_mep_dollar_rate("AL30")
        print(f"Dolar MEP obtenido: {mep_data}")
    except Exception as e:
        print(f"Error: {e}")

In [ ]:
# MEP con otros bonos
bonos_mep = ["GD30", "AL29", "AL35"]
if client:
    print("Probando dolar MEP con otros bonos...")
    for bono in bonos_mep:
        try:
            mep_data = client.get_mep_dollar_rate(bono)
            if mep_data:
                print(f"MEP {bono}: {mep_data}")
            else:
                print(f"MEP {bono}: Respuesta vacia")
        except Exception as e:
            print(f"Error MEP {bono}: {e}")

## 2. Cotizaciones de Acciones (Modelos Tipados)

In [ ]:
# Cotizaciones con modelos de objetos
acciones_test = ["GGAL", "YPFD", "PAMP", "ALUA"]

if client:
    print("Probando cotizaciones con modelos de objetos...")
    
    for accion in acciones_test:
        try:
            cotizacion = client.get_stock_quote(accion)
            
            if isinstance(cotizacion, CotizacionTitulo):
                print(f"\n{accion} - {cotizacion.descripcion_titulo}")
                print(f"  Precio: ${cotizacion.ultimo_precio}")
                print(f"  Variacion: {cotizacion.variacion}%")
                print(f"  Apertura: ${cotizacion.apertura} | Max: ${cotizacion.maximo} | Min: ${cotizacion.minimo}")
                print(f"  Spread: {cotizacion.spread}")
                print(f"  Puntas: {len(cotizacion.puntas) if cotizacion.puntas else 0}")
        except Exception as e:
            print(f"Error {accion}: {e}")

In [ ]:
# Cotizaciones con diferentes plazos
if client:
    print("Probando cotizaciones con diferentes plazos...")
    
    plazos_test = [
        (SettlementTerms.T0, "T0 - Inmediata"),
        (SettlementTerms.T1, "T1 - 1 dia"),
        (SettlementTerms.T2, "T2 - 2 dias"),
        (SettlementTerms.T3, "T3 - 3 dias")
    ]
    
    for plazo, descripcion in plazos_test:
        try:
            cotizacion = client.get_stock_quote('GGAL', settlement_term=plazo)
            print(f"GGAL {descripcion}: ${cotizacion.ultimo_precio} | Var: {cotizacion.variacion}%")
        except Exception as e:
            print(f"Error GGAL {descripcion}: {e}")

In [ ]:
# Cotizaciones en diferentes mercados
if client:
    print("Probando cotizaciones en diferentes mercados...")
    
    mercados_test = [
        (Markets.BCBA, "GGAL", "Bolsa Buenos Aires"),
        (Markets.NYSE, "AAPL", "New York Stock Exchange"),
        (Markets.NASDAQ, "MSFT", "NASDAQ")
    ]
    
    for mercado, simbolo, descripcion in mercados_test:
        try:
            cotizacion = client.get_stock_quote(simbolo, market=mercado)
            print(f"{simbolo} en {descripcion}: ${cotizacion.ultimo_precio} ({cotizacion.moneda})")
        except Exception as e:
            print(f"{simbolo} en {descripcion}: No disponible o requiere permisos")

## 3. Cotizaciones (Formato JSON)

In [ ]:
# Cotizaciones en formato JSON
if client:
    print("Probando cotizaciones en formato JSON...")
    
    for accion in ["GGAL", "YPFD"]:
        try:
            quote_data = client.get_stock_quote_raw(accion)
            print(f"\n{accion}:")
            print(f"  ultimoPrecio: {quote_data.get('ultimoPrecio')}")
            print(f"  variacionPorcentual: {quote_data.get('variacionPorcentual')}")
            print(f"  volumenNominal: {quote_data.get('volumenNominal')}")
        except Exception as e:
            print(f"Error {accion}: {e}")

## 4. Datos de Titulos

In [ ]:
# Datos de titulos
if client:
    print("Probando datos de titulos...")
    
    for simbolo in ["ALUA", "GGAL", "PAMP"]:
        try:
            datos = client.get_stock_data(simbolo)
            print(f"\n{simbolo}:")
            print(f"  Descripcion: {datos.descripcion}")
            print(f"  Pais: {datos.pais} | Mercado: {datos.mercado}")
            print(f"  Tipo: {datos.tipo} | Moneda: {datos.moneda}")
        except Exception as e:
            print(f"Error {simbolo}: {e}")

## 5. Opciones de Titulos

In [ ]:
# Opciones de titulos
if client:
    print("Probando opciones de titulos...")
    print("NOTA: No todos los titulos tienen opciones disponibles")
    
    try:
        opciones = client.get_stock_options("GGAL")
        
        if opciones:
            print(f"\nEncontradas {len(opciones)} opciones para GGAL")
            
            calls = [opt for opt in opciones if opt.es_call]
            puts = [opt for opt in opciones if opt.es_put]
            print(f"  Calls: {len(calls)} | Puts: {len(puts)}")
            
            # Mostrar primeras 3 opciones
            print("\n  Primeras 3 opciones:")
            for i, opcion in enumerate(opciones[:3], 1):
                precio = opcion.cotizacion.ultimo_precio if opcion.cotizacion else 0
                print(f"    {i}. {opcion.tipo_opcion} Strike {opcion.precio_strike}: ${precio}")
                print(f"       Vencimiento: {opcion.fecha_vencimiento.strftime('%d/%m/%Y')}")
        else:
            print("No se encontraron opciones")
            
    except Exception as e:
        print(f"Error: {e}")

## 6. Instrumentos por Pais

In [ ]:
# Instrumentos por pais
if client:
    print("Probando instrumentos por pais...")
    
    paises = [("argentina", "Argentina"), ("estados_unidos", "Estados Unidos")]
    
    for pais, descripcion in paises:
        try:
            instrumentos = client.get_market_instruments(pais)
            if instrumentos:
                print(f"\n{descripcion}: {len(instrumentos)} instrumentos")
                for inst in instrumentos:
                    print(f"  - {inst.instrumento}")
        except Exception as e:
            print(f"Error {descripcion}: {e}")

## Metodos RAW Disponibles

In [ ]:
print("METODOS RAW DE COTIZACIONES BASICAS")
print("="*50)
print("""
Para obtener respuestas en formato JSON crudo:

- client.get_stock_quote_raw(symbol, market, settlement_term)
- client.get_stock_data_raw(symbol, market)
- client.get_stock_options_raw(symbol, market)
- client.get_market_instruments_raw(pais)

Estos metodos retornan el JSON exacto de la API de IOL.
""")

## Limpieza

In [ ]:
if client:
    try:
        client.close()
        print("Cliente IOL cerrado correctamente")
    except Exception as e:
        print(f"Error al cerrar cliente: {e}")